# PCB Defect Training Pipeline (Kaggle)

Notebook nay huong dan train day du tren Kaggle cho ca YOLO va non-YOLO (MMDetection) voi cach giam xung dot Python 3.12/OpenXLab.

Quy trinh:
1. Kiem tra moi truong va duong dan
2. Cai dependencies theo che do an toan
3. Train qua scripts/train_kaggle.py
4. Kiem tra ket qua: model, metrics, plots, zip trong /kaggle/working/result

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print('\n[CMD]', ' '.join(shlex.quote(c) for c in cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None, env=env)

def kaggle_clean_env() -> dict[str, str]:
    env = os.environ.copy()
    env.pop('PYTHONPATH', None)
    env.pop('PYTHONHOME', None)
    env['PYTHONNOUSERSITE'] = '1'
    env['MIM_DISABLE_OPENXLAB'] = '1'
    return env

print('Python:', sys.version)
print('Kaggle runtime:', Path('/kaggle').exists())
print('CWD:', Path.cwd())

In [ ]:
# Cau hinh chinh
CONFIG = {
    # Thu muc code ban da upload len Kaggle (khong clone github)
    'repo_dir': '/kaggle/working/PCB_defect_detection',

    # Neu ban mount dataset dung cau truc data/train|val|test + annotations_json
    'data_source': '/kaggle/input/pcb-defect-detection/data',

    # Danh sach model can train
    'models': ['yolo11s', 'retinanet', 'faster_rcnn'],

    # Hyperparameters
    'epochs': 25,
    'early_stop': 4,
    'imgsz': 640,
    'batch': 16,
    'workers': 8,
    'gpus': 2,
    'yolo_device': '0,1',

    # Output
    'project': '/kaggle/working/runs',
    'checkpoint_root': '/kaggle/working/checkpoints',
    'result_root': '/kaggle/working/result',

    # Dependency strategy
    # True: bo qua openmim, giam kha nang conflict openxlab
    'skip_openmim': True,

    # MMDet runner: 'module' khuyen nghi tren Kaggle
    'mmdet_runner': 'module',

    # Neu dung python rieng trong venv thi dien vao day, neu khong de None
    'mmdet_python': None,

    # Co cai dependencies truoc khi train hay khong
    'install_deps': True,
}

CONFIG

In [ ]:
# Chuan bi source code local (khong clone)
repo_dir = Path(CONFIG['repo_dir'])

# Tu dong thu cac vi tri pho bien neu repo_dir khong ton tai
if not repo_dir.exists():
    candidates = [
        Path.cwd(),
        Path('/kaggle/working/PCB_defect_detection'),
        Path('/kaggle/input/pcb-defect-detection/PCB_defect_detection'),
    ]
    found = None
    for c in candidates:
        if (c / 'scripts' / 'train_kaggle.py').exists():
            found = c
            break
    if found is not None:
        repo_dir = found

train_script = repo_dir / 'scripts' / 'train_kaggle.py'
if not train_script.exists():
    raise FileNotFoundError(
        f'Khong tim thay train script tai {train_script}. Vui long upload day du source code project len Kaggle.'
    )

print('Repo dir:', repo_dir)
print('Train script:', train_script)

In [ ]:
# Cai dependencies
# Ghi chu:
# - skip_openmim=True thuong on dinh hon tren Kaggle Python 3.12
# - Neu can mmcv qua mim, dat skip_openmim=False
if CONFIG['install_deps']:
    run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
    run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_dir / 'scripts' / 'requirements_train.txt')])

    if CONFIG['skip_openmim']:
        run([sys.executable, '-m', 'pip', 'install', 'mmcv>=2.1.0,<2.2.0'])
    else:
        run([sys.executable, '-m', 'pip', 'install', '-U', 'openmim'])
        run([sys.executable, '-m', 'mim', 'install', 'mmcv>=2.1.0,<2.2.0'], env=kaggle_clean_env())

print('Dependency step done')

In [ ]:
# Ham train tung model
def build_train_cmd(model_name: str, cli_overrides: list[str] | None = None) -> list[str]:
    cmd = [
        sys.executable,
        str(repo_dir / 'scripts' / 'train_kaggle.py'),
        '--data-source', CONFIG['data_source'],
        '--project', CONFIG['project'],
        '--checkpoint-root', CONFIG['checkpoint_root'],
        '--result-root', CONFIG['result_root'],
        '--epochs', str(CONFIG['epochs']),
        '--early-stop', str(CONFIG['early_stop']),
        '--imgsz', str(CONFIG['imgsz']),
        '--batch', str(CONFIG['batch']),
        '--workers', str(CONFIG['workers']),
        '--gpus', str(CONFIG['gpus']),
        '--yolo-device', CONFIG['yolo_device'],
        '--mmdet-runner', CONFIG['mmdet_runner'],
        '--models', model_name,
    ]

    if CONFIG['install_deps']:
        cmd.append('--install-deps')

    if CONFIG['skip_openmim']:
        cmd.append('--skip-openmim')

    if CONFIG['mmdet_python']:
        cmd.extend(['--mmdet-python', CONFIG['mmdet_python']])

    if cli_overrides:
        cmd.extend(cli_overrides)

    return cmd

def train_one_model(model_name: str, cli_overrides: list[str] | None = None) -> None:
    result_root = Path(CONFIG['result_root'])
    result_root.mkdir(parents=True, exist_ok=True)
    before = {p.name for p in result_root.glob('*.zip')}

    cmd = build_train_cmd(model_name, cli_overrides=cli_overrides)
    print(f'Train command ({model_name}):')
    print(' '.join(shlex.quote(c) for c in cmd))
    run(cmd, cwd=repo_dir, env=kaggle_clean_env())

    after = sorted(result_root.glob('*.zip'))
    new_zips = [p for p in after if p.name not in before]
    model_zips = [p for p in new_zips if p.name.startswith(f'{model_name}_')]
    if not model_zips:
        # fallback khi zip khong nam trong delta (vi du cell rerun), lay zip moi nhat cua model
        model_zips = sorted(result_root.glob(f'{model_name}_*.zip'))[-1:]

    if not model_zips:
        print(f'[WARN] Chua tim thay zip ket qua cho model {model_name}')
        return

    latest_zip = sorted(model_zips, key=lambda p: p.stat().st_mtime)[-1]
    pkg_dir = result_root / latest_zip.stem
    plot_file = pkg_dir / 'plots' / 'metrics_overview.png'
    metrics_file = pkg_dir / 'metrics_summary.json'

    print(f'[OK] Zip model: {latest_zip}')
    print(f'     Plot:    {plot_file} (exists={plot_file.exists()})')
    print(f'     Metrics: {metrics_file} (exists={metrics_file.exists()})')

In [ ]:
# Override sieu tham so ngay tren notebook (giong cach go CLI)
# Vi du: ['--epochs', '30'] hoac ['--epochs', '30', '--batch', '12', '--imgsz', '768']
CLI_OVERRIDES = []

# Vi du dung python rieng tren Kaggle:
# CLI_OVERRIDES = ['--epochs', '30', '--mmdet-python', '/kaggle/working/venvs/pcb_env/bin/python']

CLI_OVERRIDES

## Train Từng Model (mỗi cell một model)

Chay cell nao thi train model do. Neu khong can thi bo qua cell do.

In [ ]:
# Cell train YOLO
train_one_model('yolo11s', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Cell train RetinaNet
train_one_model('retinanet', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Cell train Faster R-CNN
train_one_model('faster_rcnn', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Cell train Cascade R-CNN
train_one_model('cascade_rcnn', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Cell train DETR
train_one_model('detr', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Cell train Deformable DETR
train_one_model('deformable_detr', cli_overrides=CLI_OVERRIDES)

In [ ]:
# Liet ke ket qua zip trong /kaggle/working/result
result_root = Path(CONFIG['result_root'])
zip_files = sorted(result_root.glob('*.zip'))
print('Result root:', result_root)
print('Total zip files:', len(zip_files))
for p in zip_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f'- {p.name}: {size_mb:.2f} MB')

In [ ]:
# Doc nhanh metrics summary cua package moi nhat
packages = sorted([p for p in result_root.iterdir() if p.is_dir()])
if not packages:
    print('Khong tim thay package folder nao trong result_root')
else:
    latest = packages[-1]
    metrics_file = latest / 'metrics_summary.json'
    print('Latest package:', latest)
    if metrics_file.exists():
        data = json.loads(metrics_file.read_text(encoding='utf-8'))
        print(json.dumps(data, indent=2))
    else:
        print('Khong tim thay metrics_summary.json')

## Ghi chu quan trong

- Neu train non-YOLO bi loi lien quan openxlab, giu `skip_openmim=True` va `mmdet_runner='module'`.
- Neu can dung moi truong Python rieng (venv), tao truoc va set `mmdet_python` den duong dan python trong venv (vi du: `/kaggle/working/venvs/pcb_env/bin/python`).
- Sau moi model, script se tu tao metric summary + plot + zip trong `/kaggle/working/result`.